In [ ]:
## ------------------------------------------------------------------
## Setup
## ------------------------------------------------------------------
# Install/load packages you’ll need here (safe to re-run)
req_pkgs <- c("devtools", "ggplot2", "dplyr", "tidyr", "tibble", "scales")
need <- setdiff(req_pkgs, rownames(installed.packages()))
if (length(need)) install.packages(need, dependencies = TRUE)
invisible(lapply(req_pkgs, require, character.only = TRUE))

## Load your package code
devtools::load_all("/data/muscat_data/jaguir26/exdqlm")

## ------------------------------------------------------------------
## Helpers: checks, tidy df, plotting (clear colors, one plot/scenario)
## ------------------------------------------------------------------

# Basic validation of simulator output
assert_out_ok <- function(out, scenario) {
  stopifnot(is.list(out), is.numeric(out$y), is.matrix(out$q))
  T <- length(out$y); K <- ncol(out$q)
  if (nrow(out$q) != T) stop("q must be T x K.")
  mono_ok <- all(apply(out$q, 1L, function(r) all(diff(r) >= -1e-10)))
  if (!mono_ok) stop("Row-wise quantile monotonicity violated.")
  message(sprintf("OK: shapes T=%d, K=%d; monotonicity passed. (%s)", T, K, scenario))
  invisible(TRUE)
}

# Build long data.frame with selected quantiles as lines
make_quantile_df <- function(out, t_range = NULL,
                             p_lines = NULL,  # NULL -> use all out$p
                             include_mu = TRUE) {
  T <- length(out$y)
  idx <- if (is.null(t_range)) seq_len(T) else as.integer(t_range)
  idx <- idx[idx >= 1 & idx <= T]

  # which quantiles to plot (use nearest available to requested p_lines)
  if (is.null(p_lines)) {
    sel_idx <- seq_along(out$p)
    p_sel   <- out$p
  } else {
    p_lines <- sort(unique(p_lines))
    sel_idx <- vapply(p_lines, function(pp) which.min(abs(out$p - pp)), integer(1))
    p_sel   <- out$p[sel_idx]
  }

  # wide -> long
  Qsel <- as.data.frame(out$q[idx, sel_idx, drop = FALSE])
  colnames(Qsel) <- sprintf("p_%g", p_sel)

  base <- data.frame(t = idx, y = out$y[idx])
  if (include_mu && !is.null(out$extras) && !is.null(out$extras$mu)) {
    base$mu <- out$extras$mu[idx]
  }

  df_long <- dplyr::bind_cols(base, Qsel) |>
    tidyr::pivot_longer(dplyr::starts_with("p_"),
                        names_to = "p_lab", values_to = "q") |>
    dplyr::mutate(p = as.numeric(sub("^p_", "", .data$p_lab))) |>
    dplyr::select(-p_lab)

  df_long
}

# Utility: central band data (e.g., 10–90%)
make_band_df <- function(df_long, band = c(0.10, 0.90)) {
  band <- sort(band)
  if (!all(band > 0 & band < 1)) return(NULL)
  avail <- sort(unique(df_long$p))
  lo <- band[1]; hi <- band[2]
  # nearest available
  lo_use <- avail[which.min(abs(avail - lo))]
  hi_use <- avail[which.min(abs(avail - hi))]
  if (abs(lo_use - hi_use) < 1e-12) return(NULL)

  df_long |>
    dplyr::filter(p %in% c(lo_use, hi_use)) |>
    dplyr::select(t, p, q) |>
    tidyr::pivot_wider(names_from = p, values_from = q) |>
    dplyr::rename(q_lo = !!as.name(as.character(lo_use)),
                  q_hi = !!as.name(as.character(hi_use)))
}

# Plot observed y with multiple quantile lines (and optional true mu + central band)
plot_quantile_lines <- function(df_long,
                                median_p = 0.50,
                                title = NULL,
                                band = c(0.10, 0.90),   # set NULL to disable
                                palette = "Viridis") { # base grDevices palette

  # identify median rows to style them thicker
  df_med <- dplyr::filter(df_long, abs(.data$p - median_p) < 1e-12)

  # color palette for p (numeric)
  p_vals <- sort(unique(df_long$p))
  pal_cols <- grDevices::hcl.colors(100, palette = palette)
  col_scale <- ggplot2::scale_color_gradientn(
    colors = pal_cols,
    limits = c(min(p_vals), max(p_vals)),
    labels = scales::percent_format(accuracy = 1),
    name   = "quantile p"
  )

  # Optional subtle band between two quantiles (e.g., 10–90%)
  band_df <- NULL
  if (!is.null(band)) band_df <- make_band_df(df_long, band = band)

  g <- ggplot2::ggplot(df_long, ggplot2::aes(x = .data$t)) +
    ggplot2::theme_minimal(base_size = 13) +
    ggplot2::theme(
      panel.grid.minor = ggplot2::element_blank(),
      legend.position = "right"
    ) +
    ggplot2::labs(x = "time", y = "value", title = title)

  # central band first (under everything)
  if (!is.null(band_df)) {
    g <- g + ggplot2::geom_ribbon(
      data = band_df,
      ggplot2::aes(x = .data$t, ymin = .data$q_lo, ymax = .data$q_hi),
      inherit.aes = FALSE,
      fill = "grey60", alpha = 0.18
    )
  }

  # observed series
  g <- g + ggplot2::geom_line(ggplot2::aes(y = .data$y),
                              linewidth = 0.9, color = "black")

  # all quantile lines (colored by p)
  g <- g + ggplot2::geom_line(
    ggplot2::aes(y = .data$q, color = .data$p, group = .data$p),
    linewidth = 0.8, alpha = 0.95
  ) + col_scale

  # thicker median line (draw on top)
  if (nrow(df_med)) {
    g <- g + ggplot2::geom_line(
      data = df_med,
      ggplot2::aes(y = .data$q, group = NULL),
      linewidth = 1.4, color = "black", alpha = 0.95
    )
  }

  # true mu (if provided)
  if ("mu" %in% names(df_long)) {
    g <- g + ggplot2::geom_line(
      ggplot2::aes(y = .data$mu),
      linewidth = 1.0, linetype = 2, color = "#2B6CB0"
    )
  }

  g
}

# Calibration/coverage table and plot: proportion(y <= q_p) vs p
coverage_table <- function(out) {
  data.frame(
    p   = out$p,
    cov = colMeans(matrix(out$y, nrow = length(out$y), ncol = length(out$p)) <= out$q)
  )
}

plot_calibration <- function(out, title = "Calibration (hit-rate)") {
  tab <- coverage_table(out)
  ggplot2::ggplot(tab, ggplot2::aes(x = p, y = cov)) +
    ggplot2::geom_abline(slope = 1, intercept = 0, linewidth = 0.7, alpha = 0.6) +
    ggplot2::geom_point(size = 2.2, color = "black") +
    ggplot2::geom_line(linewidth = 0.7, color = "#2B6CB0") +
    ggplot2::coord_equal(xlim = c(min(out$p), max(out$p)),
                         ylim = c(min(out$p), max(out$p))) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 13) +
    ggplot2::theme(panel.grid.minor = ggplot2::element_blank()) +
    ggplot2::labs(x = "target p", y = "empirical P{y_t ≤ Q_p}", title = title)
}

## ------------------------------------------------------------------
## NEW: builders + saver for time series (observations + quantiles)
## ------------------------------------------------------------------

# Wide format: t, y, (optional) mu, q_005, q_010, ..., q_950
build_series_wide <- function(out) {
  stopifnot(is.list(out), is.numeric(out$y), is.matrix(out$q))
  T <- length(out$y)
  p <- out$p
  labs <- sprintf("q_%03d", round(100 * p))  # zero-padded
  Q <- as.data.frame(out$q)
  colnames(Q) <- labs
  df <- tibble::tibble(t = seq_len(T), y = out$y)
  if (!is.null(out$extras) && !is.null(out$extras$mu)) {
    df$mu <- out$extras$mu
  }
  dplyr::bind_cols(df, Q)
}

# Long format: t, p, q (+ y, optional mu for convenience)
build_series_long <- function(out) {
  wide <- build_series_wide(out)
  # gather only q_* columns
  q_cols <- grep("^q_\\d{3}$", names(wide), value = TRUE)
  tibble::as_tibble(wide) |>
    tidyr::pivot_longer(all_of(q_cols), names_to = "q_lab", values_to = "q") |>
    dplyr::mutate(p = as.numeric(sub("^q_(\\d{3})$", "\\1", .data$q_lab)) / 100) |>
    dplyr::select(t, p, q, dplyr::any_of(c("y", "mu")))
}

# Save all artifacts for a scenario
save_sim_outputs <- function(out, scenario, dir) {
  if (!dir.exists(dir)) dir.create(dir, recursive = TRUE)
  # data frames
  df_wide <- build_series_wide(out)
  df_long <- build_series_long(out)
  # write CSVs
  utils::write.csv(df_wide, file = file.path(dir, "series_wide.csv"), row.names = FALSE)
  utils::write.csv(df_long, file = file.path(dir, "series_long.csv"), row.names = FALSE)
  # RDS of the raw object
  saveRDS(out, file = file.path(dir, "sim_output.rds"))
  # meta text
  meta <- list(
    scenario = scenario,
    p = out$p,
    R_mc = out$info$R_mc,
    burnin = out$info$burnin,
    seed = out$info$seed,
    params = out$info$params
  )
  sink(file.path(dir, "meta.txt")); on.exit(sink(), add = TRUE)
  cat("Simulation metadata\n--------------------\n")
  cat("scenario: ", scenario, "\n", sep = "")
  cat("seed:     ", meta$seed, "\n", sep = "")
  cat("R_mc:     ", meta$R_mc, "\n", sep = "")
  cat("burnin:   ", meta$burnin, "\n", sep = "")
  cat("p grid:   ", paste(round(100 * meta$p), collapse = ", "), "\n", sep = "")
  cat("params:\n"); utils::str(meta$params, give.attr = FALSE)
}

## ------------------------------------------------------------------
## Run scenarios — one plot per scenario (clear colors) + SAVE SERIES
## ------------------------------------------------------------------

scenarios <- c("ar1_exal","sin_exal","hetero_exal","regime_exal","ar1_t")

run_one <- function(scn,
                    T = 5000L, burnin = 2000L, R_mc = 5000L, seed = 123,
                    # plot a curated set to keep legible; add more if you like
                    p_lines = c(0.05, 0.50, 0.95),
                    window_last = 600L,
                    save_dir = NULL,          # plots dir
                    save_data_dir = NULL,     # data dir (CSV + RDS); default: save_dir/data
                    band = c(0.10, 0.90),     # central band; set NULL to disable
                    palette = "Viridis") {

  message("Running scenario: ", scn)
  out <- simulate_ts_mc_quantiles(
    T = T, scenario = scn, R_mc = R_mc, burnin = burnin, seed = seed,
    keep_latents = TRUE, keep_draws = FALSE
  )
  assert_out_ok(out, scn)

  # focus on last window (post-transient)
  t_last <- (T - window_last + 1):T

  df_long <- make_quantile_df(out, t_range = t_last,
                              p_lines = p_lines, include_mu = TRUE)

  p_ts  <- plot_quantile_lines(df_long, median_p = 0.50,
                               title = paste0(scn, " — last ", window_last, " points"),
                               band = band, palette = palette)
  p_cal <- plot_calibration(out, title = paste0(scn, " — calibration"))

  print(p_ts)
  print(p_cal)

  # ----- save figures (optional)
  if (!is.null(save_dir)) {
    if (!dir.exists(save_dir)) dir.create(save_dir, recursive = TRUE)
    ggplot2::ggsave(filename = file.path(save_dir, paste0("ts_", scn, ".png")),
                    plot = p_ts, width = 11, height = 5.5, dpi = 220)
    ggplot2::ggsave(filename = file.path(save_dir, paste0("calib_", scn, ".png")),
                    plot = p_cal, width = 6.5, height = 5.5, dpi = 220)
  }

  # ----- save data (observations + quantiles + meta + RDS)
  if (is.null(save_data_dir) && !is.null(save_dir)) {
    save_data_dir <- file.path(save_dir, "data")
  }
  if (!is.null(save_data_dir)) {
    # put each scenario in its own subfolder
    scen_dir <- file.path(save_data_dir, scn)
    save_sim_outputs(out, scn, scen_dir)
  }

  invisible(list(out = out, df_long = df_long, p_ts = p_ts, p_cal = p_cal))
}

# Run them (prints one pair of plots per scenario)
# Set a base folder to collect plots + data (optional)
base_out <- "/data/muscat_data/jaguir26/exdqlm/results/sim_suite"
res_list <- lapply(
  scenarios,
  run_one,
  save_dir = file.path(base_out, "plots"),
  save_data_dir = file.path(base_out, "series")
)

## ------------------------------------------------------------------
## Done
## ------------------------------------------------------------------
# You’ll find per-scenario CSVs + RDS under:
#   /data/muscat_data/jaguir26/exdqlm/results/sim_suite/series/<scenario>/
# and PNGs under:
#   /data/muscat_data/jaguir26/exdqlm/results/sim_suite/plots/


In [ ]:
## ------------------------------------------------------------------
## Setup
## ------------------------------------------------------------------
# Install/load packages you’ll need here (safe to re-run)
req_pkgs <- c("devtools", "ggplot2", "dplyr", "tidyr", "tibble", "scales", "Matrix")
need <- setdiff(req_pkgs, rownames(installed.packages()))
if (length(need)) install.packages(need, dependencies = TRUE)
invisible(lapply(req_pkgs, require, character.only = TRUE))

## Load your package code
devtools::load_all("/data/muscat_data/jaguir26/exdqlm")

## ------------------------------------------------------------------
## Helpers: checks, tidy df, plotting (clear colors, one plot/scenario)
## ------------------------------------------------------------------

# Basic validation of simulator output
assert_out_ok <- function(out, scenario) {
  stopifnot(is.list(out), is.numeric(out$y), is.matrix(out$q))
  T <- length(out$y); K <- ncol(out$q)
  if (nrow(out$q) != T) stop("q must be T x K.")
  mono_ok <- all(apply(out$q, 1L, function(r) all(diff(r) >= -1e-10)))
  if (!mono_ok) stop("Row-wise quantile monotonicity violated.")
  message(sprintf("OK: shapes T=%d, K=%d; monotonicity passed. (%s)", T, K, scenario))
  invisible(TRUE)
}

# Build long data.frame with selected quantiles as lines
# Windowed, duplicate-safe builder: returns long df with columns t, p, q, (y, mu?)
make_quantile_df <- function(out,
                             n = NULL,                 # how many points to show
                             side = c("last","first"), # which side if n is used
                             t_range = NULL,           # explicit indices (overrides n/side)
                             p_lines = NULL,           # which ps to plot (snaps to nearest)
                             include_mu = TRUE,
                             cap = 3300,               # default window cap
                             snap_warn_tol = 1e-6) {

  stopifnot(is.list(out), is.numeric(out$y), is.matrix(out$q))
  T <- length(out$y)
  side <- match.arg(side)

  # --- choose time window ---
  if (is.null(t_range)) {
    if (is.null(n)) n <- min(T, cap)
    n <- max(1, min(n, T))
    idx <- if (side == "last") seq.int(T - n + 1L, T) else seq_len(n)
  } else {
    idx <- as.integer(t_range)
    idx <- idx[idx >= 1L & idx <= T]
  }

  # --- choose quantiles (dedup nearest matches) ---
  if (is.null(p_lines)) {
    sel_idx <- seq_along(out$p)
    p_sel   <- out$p
  } else {
    p_req <- sort(unique(as.numeric(p_lines)))
    nearest <- vapply(p_req, function(pp) which.min(abs(out$p - pp)), integer(1))
    # dedup columns so we don't get duplicate names/columns
    sel_idx <- unique(nearest)
    p_sel   <- out$p[sel_idx]

    # optional heads-up when a request snaps far from available grid
    snapped <- abs(p_sel - p_req[match(sel_idx, nearest)]) > snap_warn_tol
    if (any(snapped, na.rm = TRUE)) {
      msg <- paste0(sprintf("%g→%g", p_req[snapped], p_sel[snapped]), collapse = ", ")
      message("Note: snapped p_lines to nearest available: ", msg)
    }
  }

  # --- build long df WITHOUT relying on column names ---
  # Qsel: [length(idx) x length(sel_idx)], column-major vectorization
  Qsel <- out$q[idx, sel_idx, drop = FALSE]
  df_long <- tibble::tibble(
    t = rep(idx, times = length(sel_idx)),
    p = rep(p_sel, each = length(idx)),
    q = as.numeric(Qsel)
  )

  # add y (and mu) by join on t
  base <- tibble::tibble(t = idx, y = out$y[idx])
  if (include_mu && !is.null(out$extras) && !is.null(out$extras$mu)) {
    base$mu <- out$extras$mu[idx]
  }
  df_long <- dplyr::left_join(df_long, base, by = "t")

  if (!include_mu && "mu" %in% names(df_long)) df_long$mu <- NULL
  df_long
}

# Utility: central band data (e.g., 10–90%)
make_band_df <- function(df_long, band = c(0.10, 0.90)) {
  band <- sort(band)
  if (!all(band > 0 & band < 1)) return(NULL)
  avail <- sort(unique(df_long$p))
  lo <- band[1]; hi <- band[2]
  lo_use <- avail[which.min(abs(avail - lo))]
  hi_use <- avail[which.min(abs(avail - hi))]
  if (abs(lo_use - hi_use) < 1e-12) return(NULL)

  df_long |>
    dplyr::filter(p %in% c(lo_use, hi_use)) |>
    dplyr::select(t, p, q) |>
    tidyr::pivot_wider(names_from = p, values_from = q) |>
    dplyr::rename(q_lo = !!as.name(as.character(lo_use)),
                  q_hi = !!as.name(as.character(hi_use)))
}

# Plot observed y with multiple quantile lines (and optional true mu + central band)
plot_quantile_lines <- function(df_long,
                                median_p = 0.50,
                                title = NULL,
                                band = c(0.10, 0.90),   # set NULL to disable
                                palette = "Viridis") {

  df_med <- dplyr::filter(df_long, abs(.data$p - median_p) < 1e-12)

  p_vals <- sort(unique(df_long$p))
  pal_cols <- grDevices::hcl.colors(100, palette = palette)
  col_scale <- ggplot2::scale_color_gradientn(
    colors = pal_cols,
    limits = c(min(p_vals), max(p_vals)),
    labels = scales::percent_format(accuracy = 1),
    name   = "quantile p"
  )

  band_df <- NULL
  if (!is.null(band)) band_df <- make_band_df(df_long, band = band)

  g <- ggplot2::ggplot(df_long, ggplot2::aes(x = .data$t)) +
    ggplot2::theme_minimal(base_size = 13) +
    ggplot2::theme(panel.grid.minor = ggplot2::element_blank(),
                   legend.position = "right") +
    ggplot2::labs(x = "time", y = "value", title = title)

  if (!is.null(band_df)) {
    g <- g + ggplot2::geom_ribbon(
      data = band_df,
      ggplot2::aes(x = .data$t, ymin = .data$q_lo, ymax = .data$q_hi),
      inherit.aes = FALSE,
      fill = "grey60", alpha = 0.18
    )
  }

  g <- g + ggplot2::geom_line(ggplot2::aes(y = .data$y),
                              linewidth = 0.9, color = "black")

  g <- g + ggplot2::geom_line(
    ggplot2::aes(y = .data$q, color = .data$p, group = .data$p),
    linewidth = 0.8, alpha = 0.95
  ) + col_scale

  if (nrow(df_med)) {
    g <- g + ggplot2::geom_line(
      data = df_med,
      ggplot2::aes(y = .data$q, group = NULL),
      linewidth = 1.4, color = "black", alpha = 0.95
    )
  }

  if ("mu" %in% names(df_long)) {
    g <- g + ggplot2::geom_line(
      ggplot2::aes(y = .data$mu),
      linewidth = 1.0, linetype = 2, color = "#2B6CB0"
    )
  }

  g
}

# Calibration/coverage table and plot
coverage_table <- function(out) {
  data.frame(
    p   = out$p,
    cov = colMeans(matrix(out$y, nrow = length(out$y), ncol = length(out$p)) <= out$q)
  )
}

plot_calibration <- function(out, title = "Calibration (hit-rate)") {
  tab <- coverage_table(out)
  ggplot2::ggplot(tab, ggplot2::aes(x = p, y = cov)) +
    ggplot2::geom_abline(slope = 1, intercept = 0, linewidth = 0.7, alpha = 0.6) +
    ggplot2::geom_point(size = 2.2, color = "black") +
    ggplot2::geom_line(linewidth = 0.7, color = "#2B6CB0") +
    ggplot2::coord_equal(xlim = c(min(out$p), max(out$p)),
                         ylim = c(min(out$p), max(out$p))) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 13) +
    ggplot2::theme(panel.grid.minor = ggplot2::element_blank()) +
    ggplot2::labs(x = "target p", y = "empirical P{y_t ≤ Q_p}", title = title)
}

## ------------------------------------------------------------------
## Builders + saver for time series (observations + quantiles)
## ------------------------------------------------------------------

# Wide format
build_series_wide <- function(out) {
  stopifnot(is.list(out), is.numeric(out$y), is.matrix(out$q))
  T <- length(out$y)
  p <- out$p
  labs <- sprintf("q_%03d", round(100 * p))
  Q <- as.data.frame(out$q)
  colnames(Q) <- labs
  df <- tibble::tibble(t = seq_len(T), y = out$y)
  if (!is.null(out$extras) && !is.null(out$extras$mu)) df$mu <- out$extras$mu
  dplyr::bind_cols(df, Q)
}

# Long format
build_series_long <- function(out) {
  wide <- build_series_wide(out)
  q_cols <- grep("^q_\\d{3}$", names(wide), value = TRUE)
  tibble::as_tibble(wide) |>
    tidyr::pivot_longer(all_of(q_cols), names_to = "q_lab", values_to = "q") |>
    dplyr::mutate(p = as.numeric(sub("^q_(\\d{3})$", "\\1", .data$q_lab)) / 100) |>
    dplyr::select(t, p, q, dplyr::any_of(c("y", "mu")))
}

# Save all artifacts for a scenario
save_sim_outputs <- function(out, scenario, dir) {
  if (!dir.exists(dir)) dir.create(dir, recursive = TRUE)
  df_wide <- build_series_wide(out)
  df_long <- build_series_long(out)
  utils::write.csv(df_wide, file = file.path(dir, "series_wide.csv"), row.names = FALSE)
  utils::write.csv(df_long, file = file.path(dir, "series_long.csv"), row.names = FALSE)
  saveRDS(out, file = file.path(dir, "sim_output.rds"))
  meta <- list(
    scenario = scenario,
    p = out$p,
    R_mc = out$info$R_mc,
    burnin = out$info$burnin,
    seed = out$info$seed,
    params = out$info$params
  )
  sink(file.path(dir, "meta.txt")); on.exit(sink(), add = TRUE)
  cat("Simulation metadata\n--------------------\n")
  cat("scenario: ", scenario, "\n", sep = "")
  cat("seed:     ", meta$seed, "\n", sep = "")
  cat("R_mc:     ", meta$R_mc, "\n", sep = "")
  cat("burnin:   ", meta$burnin, "\n", sep = "")
  cat("p grid:   ", paste(round(100 * meta$p), collapse = ", "), "\n", sep = "")
  cat("params:\n"); utils::str(meta$params, give.attr = FALSE)
}

## ------------------------------------------------------------------
## Run scenarios — one plot per scenario + SAVE SERIES
## ------------------------------------------------------------------

# NEW: DLM scenarios
scenarios <- c("dlm_constV_smallW", "dlm_constV_bigW", "dlm_ar1V")

run_one <- function(scn,
                    T = 5000L, burnin = 2000L, R_mc = 5000L, seed = 123,
                    p_lines = c(0.05, 0.50, 0.95),
                    window_last = 200L,
                    save_dir = NULL,
                    save_data_dir = NULL,
                    band = c(0.10, 0.90),
                    palette = "Viridis") {

  message("Running scenario: ", scn)
  out <- simulate_ts_mc_quantiles(
    T = T, scenario = scn, R_mc = R_mc, burnin = burnin, seed = seed,
    keep_latents = TRUE, keep_draws = FALSE
  )
  assert_out_ok(out, scn)

  # last window
  t_last <- (T - window_last + 1):T
  df_long <- make_quantile_df(out, t_range = t_last, p_lines = p_lines, include_mu = TRUE)

  p_ts  <- plot_quantile_lines(df_long, median_p = 0.50,
                               title = paste0(scn, " — last ", window_last, " points"),
                               band = band, palette = palette)
  p_cal <- plot_calibration(out, title = paste0(scn, " — calibration"))

  print(p_ts); print(p_cal)

  if (!is.null(save_dir)) {
    if (!dir.exists(save_dir)) dir.create(save_dir, recursive = TRUE)
    ggplot2::ggsave(filename = file.path(save_dir, paste0("ts_", scn, ".png")),
                    plot = p_ts, width = 11, height = 5.5, dpi = 220)
    ggplot2::ggsave(filename = file.path(save_dir, paste0("calib_", scn, ".png")),
                    plot = p_cal, width = 6.5, height = 5.5, dpi = 220)
  }

  if (is.null(save_data_dir) && !is.null(save_dir)) {
    save_data_dir <- file.path(save_dir, "data")
  }
  if (!is.null(save_data_dir)) {
    scen_dir <- file.path(save_data_dir, scn)
    save_sim_outputs(out, scn, scen_dir)
  }

  invisible(list(out = out, df_long = df_long, p_ts = p_ts, p_cal = p_cal))
}

# Run them (prints one pair of plots per scenario)
base_out <- "/data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm"
res_list <- lapply(
  scenarios,
  run_one,
  save_dir = file.path(base_out, "plots"),
  save_data_dir = file.path(base_out, "series")
)

## ------------------------------------------------------------------
## Done
## ------------------------------------------------------------------
# You’ll find per-scenario CSVs + RDS under:
#   /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/<scenario>/
# and PNGs under:
#   /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/plots/
